# ASL Hackathon — Track 1: CNN (EfficientNet-B0) Transfer Learning

Goal: maximum Kaggle leaderboard accuracy.

- Backbone: `tf_efficientnet_b0_ns` from `timm` (swap to `mobilenetv3_large_100` for a faster model).
- Augmentation: rotation, color jitter, blur, cutout. NO horizontal flip (it changes sign meaning).
- Mixed precision + cosine LR. Best model is saved by val accuracy.
- TTA at inference (original + slight rescale) and writes `submission.csv` in the format `image_id,label`.

**Before running:** add the Kaggle dataset `grassknoted/asl-alphabet` to this notebook (`+ Add Data`). When the hackathon test set is released, update `TEST_DIR` in the config cell.

Turn ON the GPU accelerator (Settings → Accelerator → GPU T4 x2 or P100).

In [ ]:
import os, gc, time, math, random, json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm

print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('timm', timm.__version__)

In [ ]:
# ---------------- Config ----------------
from collections import deque

CLASSES = [chr(c) for c in range(ord('A'), ord('Z') + 1)] + ['space', 'del', 'nothing']
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

def find_dir_with_classes(base=Path('/kaggle/input'), max_depth=5):
    """BFS through /kaggle/input looking for a directory that contains both `A/` and `nothing/` subdirs."""
    if not base.exists():
        return None
    queue = deque([(base, 0)])
    while queue:
        d, depth = queue.popleft()
        try:
            if (d / 'A').is_dir() and (d / 'nothing').is_dir():
                return d
        except (OSError, PermissionError):
            pass
        if depth < max_depth:
            try:
                for c in d.iterdir():
                    if c.is_dir():
                        queue.append((c, depth + 1))
            except (OSError, PermissionError):
                pass
    return None

DATA_DIR = find_dir_with_classes()
if DATA_DIR is None:
    print('Contents of /kaggle/input:')
    for p in Path('/kaggle/input').rglob('*'):
        if p.is_dir():
            print(' ', p)
    raise FileNotFoundError('Could not auto-locate training data. Set DATA_DIR manually using the listing above.')
print(f'DATA_DIR = {DATA_DIR}')

# Try to auto-find a test folder. Override TEST_DIR by hand once the hackathon test set drops.
TEST_DIR = None
for cand in Path('/kaggle/input').rglob('*test*'):
    if not cand.is_dir():
        continue
    try:
        if any(p.suffix.lower() in {'.jpg', '.jpeg', '.png'} for p in cand.iterdir() if p.is_file()):
            TEST_DIR = cand
            break
    except (OSError, PermissionError):
        continue
print(f'TEST_DIR = {TEST_DIR}  (override manually if wrong)')

OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

CFG = dict(
    model_name='tf_efficientnet_b0_ns',  # alt: 'mobilenetv3_large_100' (faster), 'tf_efficientnet_b3_ns' (stronger, slower)
    img_size=224,
    batch_size=128,
    epochs=8,
    lr=3e-4,
    weight_decay=1e-4,
    val_fraction=0.1,
    num_workers=4,
    seed=42,
    label_smoothing=0.05,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG['seed'])

print('Device:', DEVICE)
print('Classes:', NUM_CLASSES)


In [ ]:
def scan_train(root: Path):
    rows = []
    for cls in CLASSES:
        cls_dir = root / cls
        if not cls_dir.is_dir():
            raise FileNotFoundError(f'Missing class dir: {cls_dir}')
        for p in cls_dir.iterdir():
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                rows.append({'path': str(p), 'label': CLASS_TO_IDX[cls], 'class': cls})
    return pd.DataFrame(rows)

df = scan_train(DATA_DIR)
print('Total samples:', len(df))
print(df['class'].value_counts().sort_index())

train_df, val_df = train_test_split(
    df, test_size=CFG['val_fraction'], stratify=df['label'], random_state=CFG['seed']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f'train={len(train_df)}  val={len(val_df)}')

In [ ]:
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

def build_train_tf(img_size):
    pad = int(img_size * 1.15)
    return A.Compose([
        A.LongestMaxSize(max_size=pad),
        A.PadIfNeeded(min_height=pad, min_width=pad, border_mode=0),
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.70, 1.00), ratio=(0.9, 1.1)),
        A.Rotate(limit=15, border_mode=0, p=0.7),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.05, p=0.7),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 24), hole_width_range=(8, 24), p=0.3),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def build_val_tf(img_size):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=0),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

class ASLDataset(Dataset):
    def __init__(self, df, transform):
        self.paths = df['path'].values
        self.labels = df['label'].values
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = np.array(Image.open(self.paths[i]).convert('RGB'))
        img = self.transform(image=img)['image']
        return img, int(self.labels[i])

train_tf = build_train_tf(CFG['img_size'])
val_tf = build_val_tf(CFG['img_size'])
train_ds = ASLDataset(train_df, train_tf)
val_ds = ASLDataset(val_df, val_tf)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True,
                          persistent_workers=CFG['num_workers'] > 0)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'] * 2, shuffle=False,
                        num_workers=CFG['num_workers'], pin_memory=True,
                        persistent_workers=CFG['num_workers'] > 0)
print('Loaders ready')

In [ ]:
model = timm.create_model(CFG['model_name'], pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'] * len(train_loader))
scaler = GradScaler()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'{CFG["model_name"]}: {n_params:.2f}M params')

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    loss_sum = 0.0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        with autocast():
            out = model(x)
            loss = criterion(out, y)
        loss_sum += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    return loss_sum / total, correct / total

best_acc = 0.0
ckpt_path = OUT_DIR / 'best_model.pth'
history = []

for epoch in range(CFG['epochs']):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{CFG["epochs"]}')
    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * x.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')
    train_loss = running_loss / len(train_ds)
    val_loss, val_acc = evaluate(model, val_loader)
    history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss, 'val_acc': val_acc, 'sec': time.time() - t0})
    print(f'Epoch {epoch+1}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  ({time.time()-t0:.1f}s)')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({'state_dict': model.state_dict(), 'cfg': CFG, 'classes': CLASSES, 'val_acc': val_acc}, ckpt_path)
        print(f'  -> saved (best={best_acc:.4f})')

pd.DataFrame(history).to_csv(OUT_DIR / 'history.csv', index=False)
print(f'Best val acc: {best_acc:.4f}')
print(f'Checkpoint: {ckpt_path}')

In [ ]:
# Reload best weights
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print('Loaded best weights, val_acc=', ckpt['val_acc'])

# Two-view TTA: standard + slightly tighter center crop
img_size = CFG['img_size']
tta_tf_a = build_val_tf(img_size)
tta_tf_b = A.Compose([
    A.LongestMaxSize(max_size=int(img_size * 1.1)),
    A.PadIfNeeded(min_height=int(img_size * 1.1), min_width=int(img_size * 1.1), border_mode=0),
    A.CenterCrop(height=img_size, width=img_size),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

@torch.no_grad()
def predict_paths(paths, tta=True, batch=64):
    out_labels = []
    out_probs = []
    for i in range(0, len(paths), batch):
        batch_paths = paths[i:i+batch]
        xs_a, xs_b = [], []
        for p in batch_paths:
            img = np.array(Image.open(p).convert('RGB'))
            xs_a.append(tta_tf_a(image=img)['image'])
            if tta:
                xs_b.append(tta_tf_b(image=img)['image'])
        xa = torch.stack(xs_a).to(DEVICE)
        with autocast():
            pa = torch.softmax(model(xa), dim=1)
        if tta:
            xb = torch.stack(xs_b).to(DEVICE)
            with autocast():
                pb = torch.softmax(model(xb), dim=1)
            probs = (pa + pb) / 2
        else:
            probs = pa
        idx = probs.argmax(1).cpu().numpy()
        out_labels.extend([IDX_TO_CLASS[int(j)] for j in idx])
        out_probs.append(probs.cpu().numpy())
    return out_labels, np.concatenate(out_probs, axis=0)

# Quick sanity check on val set
val_sample = val_df.sample(256, random_state=0)
preds, _ = predict_paths(val_sample['path'].tolist())
true = [IDX_TO_CLASS[int(l)] for l in val_sample['label'].values]
acc = sum(p == t for p, t in zip(preds, true)) / len(preds)
print(f'TTA val-subset accuracy: {acc:.4f}')

In [ ]:
# ---------------- Submission ----------------
# When the hackathon test set is released, set TEST_DIR above to point at it.
# `image_id` here is the filename stem (e.g. 'A_test' for 'A_test.jpg'). If the
# hackathon expects a different id format (e.g. with extension, or zero-padded
# integers), adjust the `image_id` line below.

def stem_to_id(stem: str) -> str:
    # Customise this if the hackathon's image_id format differs.
    return stem

if TEST_DIR.exists():
    test_paths = sorted([p for p in TEST_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
    print(f'Predicting {len(test_paths)} test images from {TEST_DIR}')
    labels, _ = predict_paths([str(p) for p in test_paths], tta=True)
    sub = pd.DataFrame({
        'image_id': [stem_to_id(p.stem) for p in test_paths],
        'label': labels,
    })
    sub_path = OUT_DIR / 'submission.csv'
    sub.to_csv(sub_path, index=False)
    print(sub.head())
    print(f'Wrote {sub_path}')
else:
    print(f'Test dir not found: {TEST_DIR}')
    print('Update TEST_DIR in the config cell once the hackathon hidden test set is attached as a Kaggle dataset.')

## Notes

- **Time budget:** ~8 epochs of EfficientNet-B0 on T4 takes ~30–45 min. Bump `epochs` if you have headroom.
- **Stronger model:** switch `model_name` to `tf_efficientnet_b3_ns` (or `convnext_tiny`) for ~1–2% extra accuracy and ~2× training time.
- **Submission format:** the hackathon expects `image_id,label` with label values being literal class names (`A–Z`, `space`, `del`, `nothing`). Confirm `stem_to_id` matches their naming once the test set drops.
- **Don't waste submissions:** max 5/day. Save daily slots for *meaningfully different* runs (different seed, model, or TTA strategy).